# Clase de Redes Neuronales Convolucionales (CNN)

## Parte 1: Fundamentos Teóricos

**Objetivos de esta sección:**

1. Entender qué es una convolución y por qué es útil en imágenes.
2. Dominar los conceptos de **filtro (kernel)**, **stride**, **padding** y **feature map**.
3. Calcular dimensionalidades de salida con la fórmula general.
4. Conocer la arquitectura **LeNet-5** (1998) y su impacto histórico.
5. Entender **YOLO** (You Only Look Once) como paradigma de detección de objetos.

---

## 1. ¿Por qué CNN y no MLP para imágenes?

Una imagen de 28×28 píxeles en escala de grises tiene **784 valores**. Una imagen RGB de 224×224 tiene:

$$224 \times 224 \times 3 = 150{,}528 \text{ valores}$$

Si la conectamos a una capa densa de 1000 neuronas:

$$150{,}528 \times 1000 = 150{,}528{,}000 \text{ parámetros}$$

Esto es **inviable** computacionalmente y propenso a sobreajuste. Las CNN resuelven esto con tres ideas clave:

1. **Conexiones locales (local receptive fields):** Cada neurona solo ve una región pequeña.
2. **Pesos compartidos (weight sharing):** El mismo filtro se aplica en toda la imagen.
3. **Invarianza a traslaciones:** Un gato es un gato sin importar dónde esté en la imagen.

---

## 2. La Operación de Convolución

### 2.1 Definición matemática

Dada una imagen de entrada $I$ y un filtro (kernel) $K$ de tamaño $k_h \times k_w$, la convolución 2D en la posición $(i,j)$ se define como:

$$(I * K)(i,j) = \sum_{m=0}^{k_h-1} \sum_{n=0}^{k_w-1} I(i+m, \; j+n) \cdot K(m, n)$$

En deep learning, la operación que realmente se implementa es la **correlación cruzada** (sin invertir el kernel), pero por convención se le sigue llamando "convolución":

$$S(i,j) = \sum_{m} \sum_{n} I(i+m, \; j+n) \cdot K(m, n)$$

### 2.2 Ejemplo numérico paso a paso

Imagen de entrada $5 \times 5$ y filtro $3 \times 3$:

$$
I = \begin{bmatrix}
1 & 2 & 3 & 0 & 1 \\
0 & 1 & 2 & 3 & 0 \\
1 & 0 & 1 & 2 & 3 \\
2 & 1 & 0 & 1 & 2 \\
3 & 2 & 1 & 0 & 1
\end{bmatrix}
\quad
K = \begin{bmatrix}
1 & 0 & -1 \\
1 & 0 & -1 \\
1 & 0 & -1
\end{bmatrix}
$$

**Paso 1:** Para la posición $(0,0)$:

$$S(0,0) = 1(1) + 2(0) + 3(-1) + 0(1) + 1(0) + 2(-1) + 1(1) + 0(0) + 1(-1) = -5$$

**Paso 2:** Deslizamos el filtro 1 posición a la derecha (stride=1) para posición $(0,1)$:

$$S(0,1) = 2(1) + 3(0) + 0(-1) + 1(1) + 2(0) + 3(-1) + 0(1) + 1(0) + 2(-1) = -2$$

**Resultado final (feature map $3 \times 3$):**

$$
S = \begin{bmatrix}
-5 & -2 & -3 \\
-4 & -1 & -1 \\
 0 &  1 & -3
\end{bmatrix}
$$

---

## 3. Parámetros Fundamentales

### 3.1 Filtro (Kernel)

- Matriz de pesos de tamaño $k_h \times k_w$ (típicamente $3\times3$, $5\times5$, $7\times7$).
- Cada filtro **detecta un patrón específico**: bordes horizontales, verticales, diagonales, texturas.
- En capas profundas, los filtros detectan conceptos más abstractos (ojos, ruedas, caras).
- **Número de parámetros por filtro:** $k_h \times k_w \times C_{in} + 1$ (el +1 es el bias).

### 3.2 Stride (paso)

- Número de píxeles que el filtro se desplaza en cada paso.
- `stride=1`: ventana densa (mayor resolución de salida).
- `stride=2`: reduce dimensiones a la mitad aproximadamente.
- **Stride > 1 se usa a veces en lugar de pooling** para reducir dimensionalidad.

### 3.3 Padding (relleno)

- Píxeles añadidos en los bordes de la imagen antes de convolucionar.
- **`padding='valid'`**: Sin relleno. La salida se reduce.
- **`padding='same'`**: Se añade relleno suficiente para que la salida tenga **el mismo tamaño espacial** que la entrada (si stride=1).
- Fórmula para `same` con stride=1: $P = \frac{k-1}{2}$ (para kernels impares).

**Ejemplo:** kernel $5\times5$ con `same` requiere $P = (5-1)/2 = 2$ píxeles de relleno en cada lado.

### 3.4 Feature Map

- Salida de aplicar un filtro sobre la entrada.
- Si aplicamos $F$ filtros, obtenemos $F$ feature maps apilados (uno por filtro).
- **Dimensión final:** $H_{out} \times W_{out} \times F$.

---

## 4. Fórmula General de Dimensionalidad

### 4.1 Dimensión espacial de salida

Para una convolución 2D:

$$H_{out} = \left\lfloor \frac{H_{in} - K_h + 2P_h}{S_h} \right\rfloor + 1$$

$$W_{out} = \left\lfloor \frac{W_{in} - K_w + 2P_w}{S_w} \right\rfloor + 1$$

Donde:
- $H_{in}, W_{in}$: altura y anchura de entrada.
- $K_h, K_w$: tamaño del kernel (alto, ancho).
- $P_h, P_w$: padding (alto, ancho).
- $S_h, S_w$: stride (alto, ancho).
- $\lfloor \cdot \rfloor$: función piso (redondeo hacia abajo).

### 4.2 Número de canales de salida

$$C_{out} = F \quad \text{(número de filtros usados)}$$

### 4.3 Número de parámetros de la capa

$$\text{Parámetros} = (K_h \times K_w \times C_{in} + 1) \times C_{out}$$

El $+1$ es el término de bias (uno por filtro).

### 4.4 Tabla de ejemplos resueltos

| Caso | $H_{in}$ | $K$ | $P$ | $S$ | $H_{out}$ (cálculo) | $H_{out}$ |
|------|----------|-----|-----|-----|----------------------|-----------|
| Sin padding | 28 | 5 | 0 | 1 | $(28-5+0)/1 + 1$ | **24** |
| Con `same` | 28 | 5 | 2 | 1 | $(28-5+4)/1 + 1$ | **28** |
| Con stride 2 | 28 | 5 | 0 | 2 | $\lfloor(28-5)/2\rfloor + 1$ | **12** |
| Con `same` y stride 2 | 28 | 5 | 2 | 2 | $\lfloor(28-5+4)/2\rfloor + 1$ | **14** |
| Kernel 3×3, same | 28 | 3 | 1 | 1 | $(28-3+2)/1 + 1$ | **28** |
| Kernel 3×3, valid | 28 | 3 | 0 | 1 | $(28-3+0)/1 + 1$ | **26** |
| MaxPool 2×2 | 28 | 2 | 0 | 2 | $\lfloor(28-2)/2\rfloor + 1$ | **14** |

### 4.5 Ejemplo con múltiples canales

Entrada: $28 \times 28 \times 3$ (RGB), Conv con 32 filtros de $5\times5$, `same`, stride=1:

- **Dimensión espacial:** $(28-5+4)/1 + 1 = 28 \Rightarrow 28 \times 28$
- **Canales de salida:** 32
- **Parámetros:** $(5 \times 5 \times 3 + 1) \times 32 = 76 \times 32 = 2432$
- **Salida:** $28 \times 28 \times 32$

---

## 5. Capas de Pooling

### 5.1 Max Pooling

Toma el **valor máximo** dentro de cada ventana. Es la más usada porque preserva las activaciones más fuertes (las características más relevantes).

$$\text{MaxPool}(i,j) = \max_{(m,n) \in \text{ventana}} I(i+m, \; j+n)$$

### 5.2 Average Pooling

Toma el **promedio** de la ventana. Usado en LeNet-5 original.

$$\text{AvgPool}(i,j) = \frac{1}{k_h \cdot k_w} \sum_{m,n} I(i+m, \; j+n)$$

### 5.3 Ejemplo numérico de MaxPool 2×2

$$
I = \begin{bmatrix}
1 & 3 & 2 & 4 \\
5 & 6 & 7 & 8 \\
9 & 2 & 1 & 0 \\
3 & 4 & 5 & 6
\end{bmatrix}
\xrightarrow{\text{MaxPool } 2\times2}
\begin{bmatrix}
6 & 8 \\
9 & 6
\end{bmatrix}
$$

### 5.4 Propiedades del pooling

- **No tiene parámetros aprendibles** (no hay pesos).
- Reduce dimensiones espaciales → menos cómputo en capas posteriores.
- Aporta **invarianza a pequeñas traslaciones**.
- **Fórmula de salida:** misma que convolución con $K$ = tamaño del pool.

---

## 6. Arquitectura LeNet-5 (LeCun et al., 1998)

### 6.1 Contexto histórico

- Diseñada por **Yann LeCun** para reconocer dígitos escritos a mano.
- Fue usada comercialmente por bancos en EE.UU. para leer cheques.
- Es considerada **la primera CNN exitosa** de la historia.
- Solo ~60,000 parámetros (comparado con millones en redes modernas).

### 6.2 Arquitectura detallada

Entrada: imagen **$32 \times 32 \times 1$** (escala de grises, normalizada).

| Capa | Tipo | Configuración | Salida | Parámetros |
|------|------|---------------|--------|------------|
| C1 | Conv2D | 6 filtros $5\times5$, stride=1, valid | $28 \times 28 \times 6$ | $(5\cdot5\cdot1+1)\cdot6 = 156$ |
| S2 | AvgPool | $2\times2$, stride=2 | $14 \times 14 \times 6$ | 0 |
| C3 | Conv2D | 16 filtros $5\times5$, stride=1, valid | $10 \times 10 \times 16$ | $(5\cdot5\cdot6+1)\cdot16 = 2416$ |
| S4 | AvgPool | $2\times2$, stride=2 | $5 \times 5 \times 16$ | 0 |
| C5 | Conv2D (o FC) | 120 filtros $5\times5$, valid | $1 \times 1 \times 120$ | $(5\cdot5\cdot16+1)\cdot120 = 48120$ |
| F6 | Dense | 84 neuronas, tanh | 84 | $120\cdot84+84 = 10164$ |
| Output | Dense | 10 neuronas, softmax/RBF | 10 | $84\cdot10+10 = 850$ |

**Total de parámetros:** $156 + 2416 + 48120 + 10164 + 850 = \mathbf{61{,}706}$

### 6.3 Cálculos de dimensionalidad paso a paso

**C1:** $(32-5+0)/1 + 1 = 28$ → $28 \times 28$

**S2:** $(28-2)/2 + 1 = 14$ → $14 \times 14$

**C3:** $(14-5+0)/1 + 1 = 10$ → $10 \times 10$

**S4:** $(10-2)/2 + 1 = 5$ → $5 \times 5$

**C5:** $(5-5+0)/1 + 1 = 1$ → $1 \times 1$

### 6.4 Adaptación a MNIST (28×28)

MNIST tiene imágenes de $28 \times 28$. Para usar LeNet-5 tal cual, podemos:

- **Opción A:** Hacer padding a $32 \times 32$ (fiel al original).
- **Opción B:** Usar `padding='same'` en C1 → mantiene $28 \times 28$, y luego todo lo demás se ajusta.
- **Opción C (la más simple):** Aceptar que la salida sea ligeramente distinta y ajustar la capa FC.

### 6.5 Diagrama ASCII de LeNet-5

```
Entrada          C1            S2            C3            S4           C5      F6    Salida
32x32x1  -->  28x28x6  -->  14x14x6  -->  10x10x16  -->  5x5x16  -->  1x1x120 --> 84 --> 10
              [Conv5x5]     [AvgPool]    [Conv5x5]     [AvgPool]    [Conv5x5]   [FC]  [FC]
              [6 filtros]   [2x2]        [16 filtros]  [2x2]        [120]       [tanh][softmax]
```

### 6.6 Resumen

- **2 capas convolucionales + 2 de pooling + 3 fully connected.**
- Activación **tanh** (no ReLU, que no existía en 1998).
- Usa **Average Pooling** (no Max Pooling).
- Es la base conceptual de todas las CNN modernas.

---

## 7. YOLO (You Only Look Once)

### 7.1 Diferencia fundamental: Clasificación vs Detección

| Tarea | Entrada | Salida | Ejemplo |
|-------|---------|--------|---------|
| **Clasificación** | Imagen | Una etiqueta | "Es un perro" |
| **Detección** | Imagen | Múltiples cajas + etiquetas | "Perro en (x1,y1,w1,h1), gato en (x2,y2,w2,h2)" |

### 7.2 Idea central de YOLO

En lugar de proponer regiones y clasificarlas (como R-CNN), YOLO:

1. Divide la imagen en una **cuadrícula $S \times S$** (ej. $7 \times 7$).
2. Cada celda predice **$B$ bounding boxes** (ej. $B=2$).
3. Cada bounding box tiene **5 valores:** $(x, y, w, h, \text{confianza})$.
4. Cada celda predice también **$C$ probabilidades de clase**.

### 7.3 Tensor de salida de YOLOv1

Para $S=7$, $B=2$, $C=20$ (Pascal VOC):

$$\text{Salida} = S \times S \times (B \times 5 + C) = 7 \times 7 \times (2 \times 5 + 20) = 7 \times 7 \times 30$$

### 7.4 Definición de la confianza

$$\text{Confianza} = P(\text{Objeto}) \times \text{IoU}_{\text{pred}}^{\text{truth}}$$

Donde **IoU** (Intersection over Union) mide el solapamiento entre la caja predicha y la real:

$$\text{IoU} = \frac{\text{Área de intersección}}{\text{Área de unión}}$$

### 7.5 Arquitectura de YOLOv1

- **24 capas convolucionales** (inspiradas en GoogLeNet).
- **2 capas fully connected** al final.
- Alterna convoluciones $1\times1$ (reducción de canales) y $3\times3$.
- **~millones de parámetros** (mucho mayor que LeNet).

### 7.6 Función de pérdida de YOLO

$$\lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} \left[(x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2\right]$$

$$+ \lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} \left[(\sqrt{w_i} - \sqrt{\hat{w}_i})^2 + (\sqrt{h_i} - \sqrt{\hat{h}_i})^2\right]$$

$$+ \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} (C_i - \hat{C}_i)^2 + \lambda_{\text{noobj}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{noobj}} (C_i - \hat{C}_i)^2$$

$$+ \sum_{i=0}^{S^2} \mathbb{1}_i^{\text{obj}} \sum_{c \in \text{clases}} (p_i(c) - \hat{p}_i(c))^2$$

**Componentes:**
1. Error de localización del centro $(x, y)$.
2. Error de localización del tamaño $(\sqrt{w}, \sqrt{h})$ — la raíz cuadrada penaliza menos los errores en cajas grandes.
3. Error de confianza para celdas **con** objeto.
4. Error de confianza para celdas **sin** objeto (con peso reducido $\lambda_{\text{noobj}}=0.5$).
5. Error de clasificación.

### 7.7 Ventajas de YOLO

- **Muy rápido:** Procesa la imagen en un solo forward pass.
- **Global:** Ve toda la imagen a la vez (menos falsos positivos en fondos).
- **End-to-end:** Se entrena directamente de imagen a detecciones.

### 7.8 Limitaciones

- **Difícil con objetos pequeños** (especialmente YOLOv1).
- **Limitado a $B$ objetos por celda** (no detecta bien grupos densos).
- Requiere **dataset grande** (Pascal VOC, COCO) y **GPU potente** para entrenar.

### 7.9 Nota práctica para la clase

**No entrenaremos YOLO desde cero** en esta sesión porque:
- Requiere dataset Pascal VOC (~10 GB) o COCO (~25 GB).
- Entrenamiento completo: **días** en GPU.
- En su lugar, usaremos **transfer learning** con YOLO preentrenado (Parte 4).

---

## 8. Resumen de la Parte 1

### Conceptos clave

| Concepto | Fórmula / Idea |
|----------|----------------|
| Convolución | $(I * K)(i,j) = \sum_m \sum_n I(i+m, j+n) K(m,n)$ |
| Dimensión de salida | $H_{out} = \lfloor (H_{in} - K + 2P)/S \rfloor + 1$ |
| Parámetros por capa | $(K_h \cdot K_w \cdot C_{in} + 1) \cdot C_{out}$ |
| Padding `same` | $P = (K-1)/2$ (para stride=1) |
| Max Pooling | $\max$ de la ventana |
| LeNet-5 | 61,706 parámetros, 2 Conv + 2 Pool + 3 FC |
| YOLO | $S \times S \times (B \cdot 5 + C)$ tensor de salida |

### Preguntas de autoevaluación

1. ¿Cuál es la dimensión de salida de una conv con entrada $64\times64$, kernel $3\times3$, padding=1, stride=2?
2. ¿Cuántos parámetros tiene una conv con 32 filtros de $3\times3$ sobre entrada con 16 canales (con bias)?
3. ¿Por qué LeNet-5 usa AvgPool y las redes modernas usan MaxPool?
4. En YOLO con $S=13$, $B=5$, $C=80$ (COCO), ¿cuál es el tamaño del tensor de salida?

### Transición a la Parte 2

Ahora que conocemos la teoría, implementaremos **LeNet-5 en TensorFlow/Keras** y luego en **PyTorch**, comparando sintaxis y rendimiento en la GPU de Colab.